# 🔧 Verify Your Setup

Run each cell below. Every check should show ✅. If any fail, fix the issue before proceeding.

**You need:**
- Python 3.10+
- Azure CLI authenticated (`az login`)
- A `.env` file with `FOUNDRY_PROJECT_ENDPOINT` and `FOUNDRY_MODEL`
- Dependencies installed (`pip install -r requirements.txt`)

In [1]:
# 1. Check Python version
import sys
assert sys.version_info >= (3, 10), f"Need Python 3.10+, got {sys.version}"
print(f"✅ Python {sys.version_info.major}.{sys.version_info.minor}")

✅ Python 3.14


In [2]:
# 2. Check dependencies and MAF imports
import importlib

for pkg in ["agent_framework", "azure.identity", "pydantic", "dotenv"]:
    importlib.import_module(pkg)
    print(f"✅ {pkg} installed")

# Verify correct MAF imports (these are the APIs we'll use in all challenges)
from agent_framework import (
    Agent, Executor, handler, response_handler,
    WorkflowBuilder, WorkflowContext, WorkflowRunState,
    AgentExecutorResponse, tool, Case, Default,
)
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatOptions
print("✅ All MAF imports successful")

c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


✅ agent_framework installed
✅ azure.identity installed
✅ pydantic installed
✅ dotenv installed
✅ All MAF imports successful


In [3]:
# 3. Check environment variables
import os
from dotenv import load_dotenv
load_dotenv("../.env")

endpoint = os.environ.get("FOUNDRY_PROJECT_ENDPOINT")
model = os.environ.get("FOUNDRY_MODEL")

assert endpoint, "❌ FOUNDRY_PROJECT_ENDPOINT not set! Copy .env.example to .env and fill in your values."
assert model, "❌ FOUNDRY_MODEL not set!"
print(f"✅ FOUNDRY_PROJECT_ENDPOINT: {endpoint[:50]}...")
print(f"✅ FOUNDRY_MODEL: {model}")

✅ FOUNDRY_PROJECT_ENDPOINT: https://nezukohub5203288486.services.ai.azure.com/...
✅ FOUNDRY_MODEL: gpt-4o


In [4]:
# 4. Check Azure CLI authentication
from azure.identity import AzureCliCredential

credential = AzureCliCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default")
print(f"✅ Azure CLI authenticated (token acquired)")

✅ Azure CLI authenticated (token acquired)


In [5]:
# 5. End-to-end test: Create a structured agent and get a response
from pydantic import BaseModel

class TestResponse(BaseModel):
    message: str
    ready: bool

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

test_agent = Agent(
    client,
    id="setup-test",
    name="SetupTestAgent",
    instructions="Always respond with ready=true and a short friendly message.",
    default_options=OpenAIChatOptions(response_format=TestResponse),
)

response = await test_agent.run("Are we ready for the workshop?")
result = TestResponse.model_validate_json(response.text)

print(f"✅ Agent responded: {result.model_dump()}")
assert result.ready, "Agent should confirm ready"
print(f"\n🎉 ALL CHECKS PASSED — You're ready for the workshop!")

✅ Agent responded: {'message': 'Absolutely! Everything is prepared and set for the workshop.', 'ready': True}

🎉 ALL CHECKS PASSED — You're ready for the workshop!
